# Excel RAG with Azure OpenAI

This notebook demonstrates how to build a RAG (Retrieval-Augmented Generation) pipeline  
for Excel files using **Azure OpenAI** instead of the standard OpenAI API.

### What you need from Azure Portal:
1. **API Key** → Azure OpenAI resource → Keys and Endpoint
2. **Endpoint** → same page (looks like `https://your-resource.openai.azure.com/`)
3. **Deployment names** → Azure OpenAI Studio → Deployments tab
   - One deployment for embeddings (e.g. `text-embedding-3-large` or `text-embedding-ada-002`)
   - One deployment for chat (e.g. `gpt-4o` or `gpt-35-turbo`)

## Step 0 — Install Libraries

In [1]:
# Run once to install required packages
!pip install langchain-community langchain-openai chromadb openpyxl -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Step 1 — Azure Configuration

In [2]:
import os

# --- Paste your Azure credentials here ---
os.environ["AZURE_OPENAI_API_KEY"]  = "your-azure-openai-key"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://ak-rag-openai.openai.azure.com/"

# --- Deployment names from Azure OpenAI Studio → Deployments ---
# Use the exact name you gave when deploying the model, not the model name itself
AZURE_EMBEDDING_DEPLOYMENT = "text-embedding-3-large"  # your embedding deployment name
AZURE_CHAT_DEPLOYMENT      = "gpt-4o"                  # your chat deployment name

# --- API version ---
AZURE_API_VERSION = "2024-08-01-preview"

## Step 2 — Imports

The only change from the original notebook:  
`OpenAIEmbeddings` → `AzureOpenAIEmbeddings`  
`ChatOpenAI` → `AzureChatOpenAI`

In [3]:
import pandas as pd
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma   # switched from FAISS
from langchain_core.documents import Document
from IPython.display import display, Markdown

/Users/akshaytelang/Documents/GenAI/Unstructured and Multimodal Data/Unstructured Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 3 — Helper Functions

In [4]:
# Load file — supports both .csv and .xlsx
def prepare_excel(file_path, max_rows=None):
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    else:
        df = pd.read_excel(file_path)

    if max_rows:
        df = df.head(max_rows)  # limit rows to avoid memory crash

    docs = [
        Document(
            page_content="\n".join(f"{col}: {val}" for col, val in row.items()),
            metadata={"row_index": int(i)}
        )
        for i, row in df.iterrows()
    ]
    print(f"Loaded {len(docs)} rows as documents")
    return docs

In [5]:
# Chunk documents → embed with Azure → store in Chroma
def retrieval(docs, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_documents(docs)
    print(f"Created {len(chunks)} chunks")

    embeddings = AzureOpenAIEmbeddings(
        azure_deployment=AZURE_EMBEDDING_DEPLOYMENT,
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=AZURE_API_VERSION,
        chunk_size=16
    )

    # Chroma is more stable than FAISS on macOS
    db = Chroma.from_documents(chunks, embeddings)
    print("Vector store ready!")
    return db

In [6]:
# Retrieve relevant chunks and generate an answer using Azure Chat
def generate_answer(db, query, k=5):
    retrieved_docs = db.similarity_search_with_score(query, k=k)
    context = "\n".join([doc.page_content for doc, score in retrieved_docs])

    prompt = f"""
    Answer the question: {query}
    Based on this retrieved context: {context}
    If the answer is not in the context, say you don't know.
    """

    model = AzureChatOpenAI(
        azure_deployment=AZURE_CHAT_DEPLOYMENT,
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=AZURE_API_VERSION
    )
    answer = model.invoke(prompt)

    usage = answer.response_metadata.get('token_usage', {})
    print(f"Tokens used — prompt: {usage.get('prompt_tokens')} | completion: {usage.get('completion_tokens')} | total: {usage.get('total_tokens')}")

    return display(Markdown(answer.content))

## Step 4 — Run the Pipeline

In [13]:
# Load the file — start with 100 rows to test, increase once working
docs = prepare_excel("data/food_reviews_1k.csv")#, max_rows=20)
docs[:3]  # preview first 3 documents

Loaded 1000 rows as documents


[Document(metadata={'row_index': 0}, page_content='ProductId: B001E4KFG0\nScore: 5\nSummary: Good Quality Dog Food\nText: I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.'),
 Document(metadata={'row_index': 1}, page_content='ProductId: B00813GRG4\nScore: 1\nSummary: Not as Advertised\nText: Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted. Not sure if this was an error or if the vendor intended to represent the product as "Jumbo".'),
 Document(metadata={'row_index': 2}, page_content='ProductId: B000LQOCH0\nScore: 4\nSummary: "Delight" says it all\nText: This is a confection that has been around a few centuries.  It is a light, pillowy citrus gelatin with nuts - in this case Filberts. And it is cut into tiny squares and then lib

In [14]:
# Build the FAISS vector store
db_faiss = retrieval(docs)

Created 1648 chunks
Vector store ready!


In [18]:
generate_answer(db_faiss, "best gravy related reviews")

Tokens used — prompt: 396 | completion: 328 | total: 724


Based on the retrieved context, the best gravy-related reviews for the product with ProductId: B0002XIB2Y are:

1. **Best white gravy!**  
   "Nothing easier. Nothing better. Even beats grandmother's white gravy recipe. Already peppered for you. You do nothing but mix it with water. Takes only a minute or two. I highly recommend this gravy mix. It will not disappoint."  

2. **Southern-inspired deliciousness!**  
   "The gravy mix I am reviewing, PIONEER COUNTRY GRAVY is DE-LICIOUS!!! I've tried others Publix brand, McCormick, etc., but this is the BEST! BY FAR! I chose to mix 1/2 water 1/2 milk, + 2 cooked, chopped sausage links, and it was phenomenal on my [...]. BISQUICK CHEESE GARLIC BISCUITS (Like Red Lobster's), served with a small side of watercress scrambled eggs. It's super easy to make and THICK! FINGER."  

3. **Wonderful gravy!**  
   "This gravy mix is excellent... except, don't use water where called for, use milk instead, and it makes it 100% better!"  

4. **Pioneer Gravy is GREAT!**  
   "I have used Pioneer Gravy for a number of years. It is VERY easy to make and tastes great, also. Goes very well over mashed potatoes, meats, etc. Try it!"  

These reviews consistently praise the taste, ease of preparation, and versatility of Pioneer Country Gravy.

In [20]:
# Ask a question
generate_answer(db_faiss, "give me the review with low score and bad review")

Tokens used — prompt: 163 | completion: 51 | total: 214


The reviews with low scores and bad reviews for ProductId: B000UZMJZO are:

- **Score: 1, Summary:** If I could give a rating under one-star...
- **Score: 1, Summary:** Horrid.

In [10]:
print(d])

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' (3653606975.py, line 1)

In [21]:
# Test with another query
generate_answer(db_faiss, "list the reviews where people are giving some kind of suggestion or feedback for improvement")

Tokens used — prompt: 172 | completion: 146 | total: 318


From the given context, the reviews that provide suggestions or feedback for improvement are:

1. **ProductId: B002BCD2OG**  
   **Summary:** "Good, but container could be better"  
   Feedback suggests that the container could be improved.

2. **ProductId: B000G6RYNE**  
   **Summary:** "ONLY awful because SOMETIMES they are awful"  
   Feedback implies inconsistency in quality.

3. **ProductId: B000G6RYNE**  
   **Summary:** "Not quite the best..."  
   Suggests there is room for improvement to make it better.

If this does not match what you were looking for, let me know!

In [16]:
# Test the fallback — ask something unrelated
generate_answer(db_faiss, "I like chocolate")

Tokens used — prompt: 369 | completion: 33 | total: 402


Based on the retrieved context, it seems you enjoy chocolate. However, if you're seeking something specific in response, I don't have enough information to provide further details.

# How to Set Up Azure OpenAI — Step by Step

---

## Part 1: Create an Azure OpenAI Resource

1. Go to [portal.azure.com](https://portal.azure.com) and sign in
2. In the search bar at the top, type **"Azure OpenAI"** and select it
3. Click **"+ Create"**
4. Fill in the form:
   - **Subscription** → select your subscription
   - **Resource group** → create new or pick existing (e.g. `rg-openai-demo`)
   - **Region** → pick one close to you (e.g. `East US`, `West Europe`)  
     > ⚠️ Not all models are available in all regions. `East US` and `Sweden Central` have the widest model availability.
   - **Name** → give it a unique name (e.g. `my-openai-resource`) — this becomes part of your endpoint URL
   - **Pricing tier** → select `Standard S0`
5. Click **Review + Create** → then **Create**
6. Wait ~1-2 minutes for deployment to complete, then click **Go to resource**

---

## Part 2: Get Your API Key and Endpoint

1. Inside your Azure OpenAI resource, click **"Keys and Endpoint"** in the left sidebar
2. Copy **KEY 1** → this is your `AZURE_OPENAI_API_KEY`
3. Copy the **Endpoint** (looks like `https://my-openai-resource.openai.azure.com/`) → this is your `AZURE_OPENAI_ENDPOINT`

---

## Part 3: Deploy Models in Azure OpenAI Studio

You need to deploy two models — one for embeddings, one for chat.

1. Inside your resource, click **"Go to Azure OpenAI Studio"** (or go to [oai.azure.com](https://oai.azure.com))
2. In the left sidebar, click **"Deployments"**
3. Click **"+ Deploy model"** → **"Deploy base model"**

### Deploy the Embedding Model:
4. Search for and select **`text-embedding-3-large`** (or `text-embedding-ada-002` if unavailable)
5. Click **Confirm**
6. Give it a deployment name — e.g. `text-embedding-3-large` (you can keep it same as model name)
7. Click **Deploy**

### Deploy the Chat Model:
8. Click **"+ Deploy model"** again
9. Search for and select **`gpt-4o`** (or `gpt-35-turbo` if unavailable)
10. Give it a deployment name — e.g. `gpt-4o`
11. Click **Deploy**

---

## Part 4: Update This Notebook

Paste your values into **Step 1 (Configuration cell)**:

```python
os.environ["AZURE_OPENAI_API_KEY"]  = "abc123..."           # from Part 2, KEY 1
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://my-openai-resource.openai.azure.com/"  # from Part 2

AZURE_EMBEDDING_DEPLOYMENT = "text-embedding-3-large"   # deployment name from Part 3
AZURE_CHAT_DEPLOYMENT      = "gpt-4o"                   # deployment name from Part 3
```

> ⚠️ **Important:** Use the **deployment name** you set in Step 3, not the model name.  
> They can be different — e.g. model is `gpt-4o` but you named the deployment `my-gpt4o`.

---

## Troubleshooting Common Errors

| Error | Likely Cause | Fix |
|-------|-------------|-----|
| `AuthenticationError` | Wrong API key | Double-check KEY 1 from portal |
| `ResourceNotFound` | Wrong endpoint URL | Make sure URL ends with `.openai.azure.com/` |
| `DeploymentNotFound` | Wrong deployment name | Check exact name in Azure OpenAI Studio → Deployments |
| `Model not available in region` | Region limitation | Try `East US` or `Sweden Central` |

# Azure basics chatModel code exection

In [ ]:
from openai import AzureOpenAI

endpoint = "https://ak-rag-openai.openai.azure.com/"
model_name = "gpt-4o"
deployment = "gpt-4o"

subscription_key = "your-azure-openai-key"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment
)

print(response.choices[0].message.content)

In [ ]:
response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a not helpful assistant for tourisum. if anything related places asked the refuse. only answer techical things",
        },
        {
            "role": "user",
            "content": "What are techenology evolves in USA?",
        }
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment
)
from IPython.display import display, Markdown          
                                                         
display(Markdown(response.choices[0].message.content))

In [ ]:
print(response)
print("Content Filter:",
response.choices[0].content_filter_results)
print("Prompt Filter :",
response.prompt_filter_results)

## Stream the output sample code for AzureAI


In [ ]:
from IPython.display import display, Markdown   
response = client.chat.completions.create(
    stream=True,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "I am going to Paris, what should I see?",
        }
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment,
)

# for update in response:
#     if update.choices:
#         print(update.choices[0].delta.content)
        
full_response = ""                                     
handle = display(Markdown(""), display_id=True)        
                                                        
for update in response:                                
    if update.choices:                                 
        full_response += update.choices[0].delta.content or ""
        handle.update(Markdown(full_response))


# Embedding model basic code from Azure AiFoundary

In [ ]:
from openai import AzureOpenAI

endpoint = "https://ak-rag-openai.openai.azure.com/"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

response = client.embeddings.create(
    input=["first phrase","second phrase","third phrase"],
    model=AZURE_EMBEDDING_DEPLOYMENT
)

for item in response.data:
    length = len(item.embedding)
    print(
        f"data[{item.index}]: length={length}, "
        f"[{item.embedding[0]}, {item.embedding[1]}, "
        f"..., {item.embedding[length-2]}, {item.embedding[length-1]}]"
    )
print(response.usage)

# Production RAG with Azure AI Search

Replacing local Chroma with **Azure AI Search** as the vector store — suitable for large datasets in production.

In [ ]:
# Install Azure AI Search LangChain integration
!pip install langchain-community azure-search-documents azure-core -q

In [ ]:
import os

# --- Azure AI Search credentials ---
# From Azure Portal → your Search resource → Keys
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://your-search-resource.search.windows.net"
os.environ["AZURE_SEARCH_KEY"]      = "your-azure-search-admin-key"

# Index name — will be created automatically if it doesn't exist
AZURE_SEARCH_INDEX = "food-reviews-index"

In [ ]:
import pandas as pd
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import AzureSearch
from langchain_core.documents import Document
from IPython.display import display, Markdown

In [ ]:
# Build shared embeddings object
def get_embeddings():
    return AzureOpenAIEmbeddings(
        azure_deployment=AZURE_EMBEDDING_DEPLOYMENT,
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=AZURE_API_VERSION,
        chunk_size=16
    )


# Chunk + embed + push to Azure AI Search index
# Run this ONCE to index your data — re-run only when data changes
def index_documents(docs, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_documents(docs)
    print(f"Created {len(chunks)} chunks — uploading to Azure AI Search...")

    embeddings = get_embeddings()

    # Creates the index automatically if it doesn't exist
    # Uploads all chunks with their embeddings to Azure AI Search
    vector_store = AzureSearch(
        azure_search_endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
        azure_search_key=os.environ["AZURE_SEARCH_KEY"],
        index_name=AZURE_SEARCH_INDEX,
        embedding_function=embeddings.embed_query
    )
    vector_store.add_documents(chunks)
    print(f"Indexed {len(chunks)} chunks into '{AZURE_SEARCH_INDEX}'")
    return vector_store


# Load existing index from Azure AI Search — no re-embedding needed
def load_index():
    embeddings = get_embeddings()
    vector_store = AzureSearch(
        azure_search_endpoint=os.environ["AZURE_SEARCH_ENDPOINT"],
        azure_search_key=os.environ["AZURE_SEARCH_KEY"],
        index_name=AZURE_SEARCH_INDEX,
        embedding_function=embeddings.embed_query
    )
    print(f"Connected to index '{AZURE_SEARCH_INDEX}'")
    return vector_store

In [ ]:
# Retrieve from Azure AI Search + generate answer
def generate_answer(vector_store, query, k=5):
    retrieved_docs = vector_store.similarity_search_with_relevance_scores(query, k=k)
    context = "\n".join([doc.page_content for doc, score in retrieved_docs])

    prompt = f"""
    Answer the question: {query}
    Based on this retrieved context: {context}
    If the answer is not in the context, say you don't know.
    """

    model = AzureChatOpenAI(
        azure_deployment=AZURE_CHAT_DEPLOYMENT,
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        api_version=AZURE_API_VERSION
    )
    answer = model.invoke(prompt)

    usage = answer.response_metadata.get('token_usage', {})
    print(f"Tokens — prompt: {usage.get('prompt_tokens')} | completion: {usage.get('completion_tokens')} | total: {usage.get('total_tokens')}")

    return display(Markdown(answer.content))

In [ ]:
# ── FIRST TIME: index your data ──────────────────────────────
docs = prepare_data("data/food_reviews_1k.csv")       # load all 1000 rows
vector_store = index_documents(docs)                   # embed + push to Azure AI Search

# ── LATER RUNS: skip indexing, load directly ─────────────────
# vector_store = load_index()                          # uncomment after first run

# ── Query ─────────────────────────────────────────────────────
generate_answer(vector_store, "Give me the best reviews with comments")

# How to Create Azure AI Search Resource — Step by Step

---

## Part 1: Create the Search Resource

1. Go to [portal.azure.com](https://portal.azure.com) and sign in
2. In the top search bar type **"Azure AI Search"** and select it
3. Click **"+ Create"**
4. Fill in the form:
   - **Subscription** → select your subscription
   - **Resource group** → use the same one as your Azure OpenAI resource (e.g. `rg-openai-demo`)
   - **Service name** → unique name (e.g. `my-search-service`) — becomes part of the endpoint URL
   - **Region** → same region as your Azure OpenAI resource
   - **Pricing tier** → click **"Change Pricing Tier"**
     - `Free (F)` → 50MB, max 3 indexes — good for this notebook
     - `Basic (B)` → 2GB — small production apps
     - `Standard S1` → 25GB+ — production workloads
5. Click **"Review + Create"** → **"Create"**
6. Wait ~2 minutes → click **"Go to resource"**

---

## Part 2: Get Your Endpoint and Key

1. In the left sidebar click **"Keys"**
2. Copy **Primary admin key** → paste as `AZURE_SEARCH_KEY`
3. In the left sidebar click **"Overview"**
4. Copy the **URL** (e.g. `https://my-search-service.search.windows.net`) → paste as `AZURE_SEARCH_ENDPOINT`

---

## Part 3: Update the Config Cell Above

```python
os.environ["AZURE_SEARCH_ENDPOINT"] = "https://my-search-service.search.windows.net"
os.environ["AZURE_SEARCH_KEY"]      = "your-primary-admin-key"
AZURE_SEARCH_INDEX                  = "food-reviews-index"
```

> The index is **created automatically** on first run — no manual setup needed in the portal.

---

## Part 4: Run Order

| Step | Code | When |
|------|------|------|
| 1 | `docs = prepare_data(...)` | Every time data changes |
| 2 | `vector_store = index_documents(docs)` | **First time only** |
| 3 | `vector_store = load_index()` | All subsequent runs |
| 4 | `generate_answer(vector_store, query)` | Every query |

---

## Pricing

| Tier | Storage | Max Indexes | Use Case |
|------|---------|-------------|----------|
| `Free` | 50 MB | 3 | Learning / this notebook |
| `Basic` | 2 GB | 5 | Dev / small apps |
| `Standard S1` | 25 GB | 50 | Production |
| `Standard S2` | 100 GB | 200 | Large-scale production |

In [1]:
import psutil
print(f"Available RAM: {psutil.virtual_memory().available/ 1024**3:.1f} GB")

Available RAM: 3.6 GB
